In [ ]:
!pip install ultralytics deep_sort_realtime opencv-python -q

In [ ]:
import cv2
import numpy as np
from base64 import b64decode, b64encode
from IPython.display import display, Javascript
from google.colab.output import eval_js
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort

model = YOLO("yolov8n.pt")
tracker = DeepSort(max_age=30, n_init=3)
class_names = model.names

np.random.seed(42)
colors = {}
def get_color(tid):
    if tid not in colors:
        colors[tid] = tuple(int(c) for c in np.random.randint(50, 255, 3))
    return colors[tid]

print("Ready!")

Ready!


In [ ]:
def video_stream():
    js = Javascript('''
    var video, div = null, stream, captureCanvas, imgElement, labelElement;
    var pendingResolve = null, shutdown = false;

    function removeDom() {
        stream.getVideoTracks()[0].stop();
        video.remove(); div.remove();
        video = null; div = null; stream = null;
        imgElement = null; captureCanvas = null; labelElement = null;
    }

    function onAnimationFrame() {
        if (!shutdown) { window.requestAnimationFrame(onAnimationFrame); }
        if (pendingResolve) {
            var result = "";
            if (!shutdown) {
                captureCanvas.getContext('2d').drawImage(video, 0, 0, 640, 480);
                result = captureCanvas.toDataURL('image/jpeg', 0.8);
            }
            var lp = pendingResolve;
            pendingResolve = null;
            lp(result);
        }
    }

    async function createDom() {
        if (div !== null) { return stream; }
        div = document.createElement('div');
        div.style.border = '2px solid black';
        div.style.padding = '3px';
        div.style.width = '100%';
        div.style.maxWidth = '640px';
        document.body.appendChild(div);

        const modelOut = document.createElement('div');
        modelOut.innerHTML = "<span>Status: </span>";
        labelElement = document.createElement('span');
        labelElement.innerText = 'Starting...';
        labelElement.style.fontWeight = 'bold';
        modelOut.appendChild(labelElement);
        div.appendChild(modelOut);

        video = document.createElement('video');
        video.style.display = 'block';
        video.width = div.clientWidth - 6;
        video.setAttribute('playsinline', '');
        video.onclick = () => { shutdown = true; };
        stream = await navigator.mediaDevices.getUserMedia({video: true});
        div.appendChild(video);

        imgElement = document.createElement('img');
        imgElement.style.position = 'absolute';
        imgElement.style.zIndex = 1;
        imgElement.onclick = () => { shutdown = true; };
        div.appendChild(imgElement);

        const instruction = document.createElement('div');
        instruction.innerHTML =
            '<span style="color:red;font-weight:bold;">Stop karne ke liye video par click karein</span>';
        div.appendChild(instruction);
        instruction.onclick = () => { shutdown = true; };

        video.srcObject = stream;
        await video.play();

        captureCanvas = document.createElement('canvas');
        captureCanvas.width = 640;
        captureCanvas.height = 480;
        window.requestAnimationFrame(onAnimationFrame);
        return stream;
    }

    async function stream_frame(label, imgData) {
        if (shutdown) { removeDom(); shutdown = false; return ''; }
        stream = await createDom();
        if (label != "") { labelElement.innerHTML = label; }
        if (imgData != "") {
            var videoRect = video.getClientRects()[0];
            imgElement.style.top = videoRect.top + "px";
            imgElement.style.left = videoRect.left + "px";
            imgElement.style.width = videoRect.width + "px";
            imgElement.style.height = videoRect.height + "px";
            imgElement.src = imgData;
        }
        var result = await new Promise(function(resolve) { pendingResolve = resolve; });
        shutdown = false;
        return {'img': result};
    }
    ''')
    display(js)

def video_frame(label, overlay):
    return eval_js('stream_frame("{}", "{}")'.format(label, overlay))

def js_to_image(js_reply):
    jpeg_bytes = b64decode(js_reply.split(',')[1])
    arr = np.frombuffer(jpeg_bytes, dtype=np.uint8)
    return cv2.imdecode(arr, cv2.IMREAD_COLOR)

def overlay_to_dataurl(overlay):
    _, png = cv2.imencode('.png', overlay)
    return 'data:image/png;base64,' + b64encode(png.tobytes()).decode()

In [ ]:
CONF_THRESHOLD = 0.4

video_stream()                 # webcam on hoti hai (browser permission: Allow karein)
label_html = "Capturing..."
overlay_data = ""

while True:
    js_reply = video_frame(label_html, overlay_data)
    if not js_reply:
        break

    frame = js_to_image(js_reply["img"])

    # 1) Object Detection (YOLO)
    results = model(frame, verbose=False)[0]
    detections = []
    for box in results.boxes:
        conf = float(box.conf[0])
        if conf < CONF_THRESHOLD:
            continue
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        detections.append(([x1, y1, x2 - x1, y2 - y1], conf, int(box.cls[0])))

    # 2) Object Tracking (Deep SORT)
    tracks = tracker.update_tracks(detections, frame=frame)

    # 3) Boxes + labels + tracking IDs
    overlay = np.zeros((480, 640, 4), dtype=np.uint8)
    confirmed = 0
    for track in tracks:
        if not track.is_confirmed():
            continue
        confirmed += 1
        tid = track.track_id
        l, t, r, b = map(int, track.to_ltrb())
        cls = track.get_det_class()
        label = f"{class_names[cls]} ID:{tid}" if cls is not None else f"ID:{tid}"
        c = get_color(tid)
        color = (c[0], c[1], c[2], 255)

        cv2.rectangle(overlay, (l, t), (r, b), color, 2)
        cv2.putText(overlay, label, (l, max(t - 8, 15)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    overlay_data = overlay_to_dataurl(overlay)
    label_html = f"Objects tracked: {confirmed}"

<IPython.core.display.Javascript object>